In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Elastic Training on Cloud TPUs with MaxText and Pathways

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/AI-Hypercomputer/maxtext/blob/main/src/maxtext/examples/elastic_training/elastic_qwen3_demo.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/AI-Hypercomputer/maxtext/blob/main/src/maxtext/examples/elastic_training/elastic_qwen3_demo.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

| Author(s) |
| --- |
| [Ivan Nardini](https://github.com/inardini) |

## Overview

This tutorial demonstrates **elastic training** on Cloud TPUs: training that survives a slice failure in-process, without restarting the job. You will provision 3 TPU v5e slices on GKE, launch a Qwen3 0.6B pre-training run orchestrated by Pathways, terminate a worker mid-run to simulate a hardware failure, and confirm that training recovers from the last checkpoint on the same head pod, in about 40 seconds, with no job restart.

**What you'll do:**
1. Provision a GKE cluster with 3 TPU v5e slices and a CPU node (`terraform apply`)
2. Deploy the MaxText image and the elastic training JobSet via Cloud Build (`gcloud builds submit`)
3. Watch training compile and reach a steady step rate (`kubectl logs`)
4. Terminate a worker and verify in-process recovery (`kubectl delete pod`)
5. Tear down all infrastructure (`terraform destroy`)

Each step runs a single `terraform`, `gcloud`, or `kubectl` command. The same steps are available as `make` targets for terminal use; this notebook calls the underlying commands directly. The Terraform config, the Cloud Build pipeline, and the training JobSet live next to this notebook in `terraform/` and `deploy/`.

### Why elastic training

Large model training runs across many TPU slices, and at that scale individual slices fail routinely, from hardware faults, preemption, or network blips. By default a single slice failure crashes the whole job: you restart from scratch and lose both the XLA compilation time (JAX compiling the model graph for the TPU) and all progress since the last checkpoint.

Elastic training avoids the full restart. **Pathways** (the runtime that orchestrates training across slices) detects the dead slice and reports it to the training process. **MaxText**'s `elastic_retry` wraps the training step and catches that error *inside the same Python process*, rather than letting it propagate and kill the job. **Orbax** (the checkpoint library) discards any checkpoint that was interrupted mid-write, and training rewinds to the last fully committed checkpoint. Because the head process never exits, the expensive XLA recompile is skipped and recovery completes in roughly 40 seconds.

This is a multi-host workload: it requires 3 live TPU v5e slices (48 chips) plus a Pathways control plane on GKE, so it runs on a real Google Cloud project with TPU quota rather than in single-runner notebook CI.

### The recovery sequence

The diagram below traces what happens the moment a slice fails. The `commit_success` marker is the key detail: Orbax writes it only after a checkpoint is fully flushed to GCS, so a checkpoint interrupted mid-write has no marker and is safely discarded, and recovery always resumes from a complete checkpoint. You will see each of these log lines appear in Step 4.

```mermaid
sequenceDiagram
    participant T as Training Loop
    participant E as elastic_retry
    participant C as Checkpointing
    participant GCS as GCS Bucket

    T->>T: Training step N
    Note over T: Slice fails
    T-->>E: Exception caught
    E->>C: clean_up_checkpoints()
    C->>GCS: Check for commit_success marker
    C->>GCS: No marker found, delete incomplete checkpoint
    E->>T: Re-initialize training
    T->>GCS: Load last good checkpoint
    T->>T: Resume training from checkpoint
```

**Notice.** This notebook provisions real infrastructure (3 TPU v5e slices + a CPU node) at on-demand list price, so a full ~30-minute run costs roughly **$30**. (TPU v5e is ~$1.20/chip-hr. See [TPU pricing](https://cloud.google.com/tpu/pricing).) **Always run the final `terraform destroy` cell when you are done**, even if you stop halfway.

## Get started

### Prerequisites

This notebook drives Google Cloud infrastructure, so it expects to run from a machine that already has the CLIs and credentials, Cloud Shell, or a local shell with the Cloud SDK. It does not install them. You'll need:

- A Google Cloud project with billing enabled.
- **TPU v5e on-demand quota**: at least 48 chips in `us-central1`. Request the `Tpu-v5-litepod-device` quota in [IAM & Admin > Quotas](https://console.cloud.google.com/iam-admin/quotas); without it, provisioning fails.
- A [Hugging Face token](https://huggingface.co/settings/tokens) with the `read` role. Qwen3 is ungated, but the pipeline wires the token through so the same setup works for a gated model.
- CLI tools on your PATH: `terraform` (>= 1.5.0), `gcloud`, and `kubectl`.

For the supported ways to run MaxText notebooks (Colab, VS Code, local Jupyter), see the [Run MaxText Python Notebooks](https://github.com/AI-Hypercomputer/maxtext/blob/main/docs/guides/run_python_notebook.md) guide.

### Set your configuration

Edit the two **required** values below, `PROJECT_ID` and `HF_TOKEN`. The remaining variables have defaults that match `terraform/variables.tf` and rarely need changing. This is the notebook's **parameters** cell, so an automated runner can override any of these values without editing the cell.

In [ ]:
# Required: edit these two (or override them when testing).
PROJECT_ID = "your-project-id"  # your GCP project ID
HF_TOKEN = "hf_..."             # your HuggingFace read token

# Optional: defaults match terraform/variables.tf.
REGION = "us-central1"
ZONE = "us-central1-a"
CLUSTER_NAME = "maxtext-elastic-training"
BUCKET_NAME = ""  # leave empty to derive maxtext-elastic-training-PROJECT_ID below
DATASET = "glaive"  # "glaive" or "synthetic"
RUN_NAME = "elastic-qwen3-0.6b"
DEBUGGING = "false"
TPU_SPOT = False       # True = cheaper Spot (preemptible) TPUs; False = on-demand
SLICE = "2"            # which slice to terminate in the failure step
RUN_SECOND_FAILURE = False  # set True to terminate a second slice at the end
HEADLESS = ""          # set non-empty by automated test runs to skip interactive prompts

# Total seconds to wait for long-running stages before giving up (headless safety).
POLL_TIMEOUT = 1800

### Apply the configuration

This cell derives computed values (such as the bucket name), exports everything to the environment so the `terraform`, `gcloud`, and `kubectl` cells can read it, and changes into the example directory so the relative `terraform/` and `deploy/` paths resolve. It is kept separate from the parameters cell so it re-runs after an automated runner injects overrides.

In [ ]:
import os

if not BUCKET_NAME:
    BUCKET_NAME = f"maxtext-elastic-training-{PROJECT_ID}"

os.environ.update(
    PROJECT_ID=PROJECT_ID, HF_TOKEN=HF_TOKEN, REGION=REGION, ZONE=ZONE,
    CLUSTER_NAME=CLUSTER_NAME, BUCKET_NAME=BUCKET_NAME, DATASET=DATASET,
    RUN_NAME=RUN_NAME, DEBUGGING=DEBUGGING, SLICE=str(SLICE),
)

# Move into the example folder so terraform/ and deploy/ resolve. Idempotent: if we are
# already inside elastic_training/ we stay put; otherwise we look for it relative to the
# cloned MaxText repo, and stop with a clear message if it is not there.
EXAMPLE_DIR = "src/maxtext/examples/elastic_training"
if os.path.basename(os.getcwd()) != "elastic_training":
    if os.path.isdir(EXAMPLE_DIR):
        os.chdir(EXAMPLE_DIR)
    elif not os.path.isdir("terraform"):
        raise RuntimeError(
            "Run this notebook from a cloned MaxText repo: "
            "`git clone https://github.com/AI-Hypercomputer/maxtext`, then open "
            "src/maxtext/examples/elastic_training/elastic_qwen3_demo.ipynb from there."
        )
assert os.path.isdir("terraform") and os.path.isdir("deploy"), (
    f"Expected terraform/ and deploy/ in {os.getcwd()}; are you in the elastic_training folder?"
)
print(f"Configured project {PROJECT_ID}, bucket {BUCKET_NAME}")
print(f"Working directory: {os.getcwd()}")

### Authenticate

Terraform, `gcloud`, and `kubectl` all act through Application Default Credentials (ADC). In Colab this cell authenticates you interactively. On a kernel that already has ADC, a TPU VM, Cloud Shell, or a service-account runner, it detects the existing credentials and does nothing.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()
elif not HEADLESS:
    # Interactive run: open the login flow only if ADC is not already present.
    # Headless runs (HEADLESS set, e.g. automated notebook testing) skip this and
    # rely on the runner's service account / ADC.
    has_adc = os.system("gcloud auth application-default print-access-token >/dev/null 2>&1") == 0
    if not has_adc:
        !gcloud auth application-default login
    else:
        print("ADC already present; skipping login.")
else:
    print("HEADLESS set; relying on the runner service account / ADC.")

### Generate `terraform.tfvars`

Terraform reads its inputs from `terraform/terraform.tfvars`. This cell writes that file from the configuration above. (The `Makefile` generates the same file from a `.env` for terminal use; the notebook writes it directly.)

In [ ]:
tfvars = f'''project_id       = "{PROJECT_ID}"
region           = "{REGION}"
zone             = "{ZONE}"
cluster_name     = "{CLUSTER_NAME}"
cluster_version  = ""
network          = "default"
subnetwork       = "default"
tpu_machine_type = "ct5lp-hightpu-4t"
tpu_topology     = "4x4"
num_tpu_slices   = 3
nodes_per_slice  = 4
cpu_machine_type = "n2-standard-64"
num_cpu_nodes    = 1
gcs_bucket_name  = "{BUCKET_NAME}"
hf_token         = "{HF_TOKEN}"
tpu_reservation  = ""
tpu_spot         = {str(TPU_SPOT).lower()}
'''

with open("terraform/terraform.tfvars", "w") as f:
    f.write(tfvars)
print("Wrote terraform/terraform.tfvars")

## Step 1: Provision the infrastructure

`terraform apply` creates everything the run needs: the GKE cluster, the TPU and CPU node pools, a GCS bucket for checkpoints, an Artifact Registry repository, a dedicated Cloud Build service account (new projects disable the legacy default compute SA), the IAM bindings, and the Hugging Face token wired into both Secret Manager and a Kubernetes secret. The `-chdir=terraform` flag runs Terraform against the `terraform/` subfolder.

This takes about **10 minutes** and starts the ~$61/hr meter (Step 6 tears it down).

### What you should see

Terraform prints a plan, applies it, and ends with:

```
Apply complete! Resources: NN added, 0 changed, 0 destroyed.
```

> **Troubleshooting**: if Terraform fails with quota errors, check your `Tpu-v5-litepod-device` quota in IAM & Admin > Quotas.

In [ ]:
!terraform -chdir=terraform init -input=false
!terraform -chdir=terraform apply -auto-approve

## Step 2: Deploy the training job

A single Cloud Build pipeline runs five phases: build the MaxText image, install the JobSet API on the cluster, download the tokenizer to GCS, prepare the training data, and apply the training JobSet. The image builds in Cloud Build, so no local Docker is needed. The build runs as the dedicated service account Terraform created in Step 1, retrieved here from the Terraform output.

This takes about **6 minutes**.

### What you should see

A stream of Cloud Build step logs ending with:

```
DONE
...
STATUS: SUCCESS
```

In [ ]:
%%bash
set -euo pipefail
# Dedicated Cloud Build SA created by Terraform (new projects disable the default compute SA).
BUILD_SA=$(terraform -chdir=terraform output -raw cloudbuild_service_account)
gcloud builds submit --config=deploy/deploy.yaml \
  --service-account="projects/$PROJECT_ID/serviceAccounts/$BUILD_SA" \
  --substitutions=_CLUSTER_NAME=$CLUSTER_NAME,_CLUSTER_ZONE=$ZONE,_BUCKET_NAME=$BUCKET_NAME,_DATASET=$DATASET,_DEBUGGING=$DEBUGGING,_RUN_NAME=$RUN_NAME \
  --project="$PROJECT_ID"

## Step 3: Check the pods

Fetch cluster credentials and list the pods to confirm the JobSet scheduled correctly.

### What you should see

**13 pods**: 1 head pod (running the training script `main`, with the Pathways Resource Manager and the IFRT proxy as sidecars) and **12 worker pods across 3 slices** (4 workers per slice). It can take a couple of minutes for all of them to reach `Running`:

```
NAME                                 READY   STATUS    RESTARTS   AGE
pw-elastic-pathways-head-0-0-xxxxx   1/1     Running   0          2m
pw-elastic-worker-0-0-xxxxx          1/1     Running   0          2m
...
```

In [ ]:
%%bash
gcloud container clusters get-credentials "$CLUSTER_NAME" \
  --zone="$ZONE" --project="$PROJECT_ID" --quiet
kubectl get pods

### Log helpers

The next steps wait for specific markers in the head pod's log, a steady step rate, then the recovery sequence. `kubectl logs --follow` streams indefinitely and never returns, which is fine to watch by hand but hangs an automated run. These helpers poll the log until the marker appears (or `POLL_TIMEOUT` elapses) and then return, so the notebook runs cleanly both interactively and headlessly.

In [ ]:
import re
import subprocess
import time


def head_pod():
    """Return the head pod name, or "" if it is not scheduled yet."""
    out = subprocess.run(
        ["kubectl", "get", "pods", "-l", "job-name=pw-elastic-pathways-head-0",
         "-o", "jsonpath={.items[0].metadata.name}"],
        capture_output=True, text=True,
    )
    return out.stdout.strip()


def head_log(tail=200):
    """Return the last `tail` lines of the head pod main container log."""
    pod = head_pod()
    if not pod:
        return ""
    out = subprocess.run(
        ["kubectl", "logs", pod, "-c", "main", f"--tail={tail}"],
        capture_output=True, text=True,
    )
    return out.stdout


def latest_step(log_text):
    """Highest "completed step: N" seen in the log text, or -1."""
    steps = [int(m) for m in re.findall(r'completed step: (\d+)', log_text)]
    return max(steps) if steps else -1


def wait_for(predicate, what, timeout=None, interval=5):
    """Poll until predicate(log_text) is true. Returns the final log text."""
    timeout = timeout or POLL_TIMEOUT
    deadline = time.time() + timeout
    while time.time() < deadline:
        log = head_log()
        if predicate(log):
            print(f"OK: {what}")
            return log
        time.sleep(interval)
    raise TimeoutError(f"Timed out after {timeout}s waiting for: {what}")

## Step 4: Watch training start

Wait for training to compile and reach a steady step rate. The cell polls the head pod log until the step counter passes ~130, which guarantees checkpoint 100 has committed, the checkpoint recovery will restore from in the next step, then prints the latest step lines. It returns on its own, so it is safe to re-run and safe for an automated run.

### What you should see

After XLA compilation (about **2-3 minutes**), the log shows that elastic training is enabled and then a steady stream of steps:

```
Elastic utils: Elastic training enabled.
Elastic Retry Enabled
completed step: 8, seconds: 0.159, TFLOP/s/device: 43.430, loss: 220.774
completed step: 9, seconds: 0.166, TFLOP/s/device: 41.524, loss: 217.296
completed step: 10, seconds: 0.160, TFLOP/s/device: 43.230, loss: 216.126
```

From then on expect **~43 TFLOP/s/device at ~0.16s/step**.

In [ ]:
log = wait_for(lambda t: latest_step(t) >= 130,
               "training reached step 130 (checkpoint 100 committed)")
for line in [l for l in log.splitlines() if "completed step:" in l][-3:]:
    print(line)

## Step 5: Simulate a slice failure

This step injects the failure. The cell first waits for a **safe checkpoint window**: elastic retry can only recover from a fully committed checkpoint, so it waits until the step is past 140 and off the write boundary, ensuring a clean checkpoint exists and none is in flight. It then terminates a worker on the chosen slice (`SLICE`, default 2) with `kubectl delete pod --grace-period=0 --force`, an immediate SIGKILL. The hard termination is deliberate: it mimics a real hardware failure, which does not drain gracefully the way an ordinary pod deletion does. Set `SLICE` in the parameters cell to target a different slice.

In [ ]:
def in_safe_window(log_text):
    step = latest_step(log_text)
    mod = step % 100
    return step >= 140 and 40 <= mod <= 95


wait_for(in_safe_window, "safe checkpoint window (step >= 140, off the write boundary)")
step_before = latest_step(head_log())
print(f"Training at step {step_before}; terminating a worker on slice {SLICE}")

victim = subprocess.run(
    ["kubectl", "get", "pods", "-l", f"job-name=pw-elastic-worker-{SLICE}",
     "-o", "jsonpath={.items[0].metadata.name}"],
    capture_output=True, text=True,
).stdout.strip()
if not victim:
    print(f"No running worker found on slice {SLICE} yet. If you just triggered a failure, "
          "wait for recovery to finish, then re-run this cell.")
else:
    print(f"Terminating {victim} (slice {SLICE}, --grace-period=0)")
    subprocess.run(["kubectl", "delete", "pod", victim, "--grace-period=0", "--force"], check=True)

## Step 6: Verify recovery

Recovery shows up in the *same head pod log* tailed earlier, which is the point: the head process never exited. Within ~15 seconds of the termination, Pathways reports the slice down and elastic retry restores the last committed checkpoint, with no new pod and no job restart. The cell waits for the restore marker and the resumed step counter, then prints the recovery lines.

### What you should see

```
Slice down event detected. Retrying.
Found commit_success file. Keeping gs://.../checkpoints/100/.
Elastic attempt 2 out of 10
Restoring checkpoint from gs://.../checkpoints/100.
completed step: 101, ...
```

The step counter dropping (e.g. 150 -> 101) is the rewind to the last committed checkpoint.

In [ ]:
log = wait_for(
    lambda t: "Restoring checkpoint from" in t or "Slice down event detected" in t,
    "recovery started (slice down detected / checkpoint restoring)",
)
markers = ("Slice down event", "Elastic attempt", "Keeping gs://",
           "Restoring checkpoint from")
for line in log.splitlines():
    if any(m in line for m in markers):
        print(line)
print(f"Latest step after recovery: {latest_step(head_log())}")

## Step 7: Confirm nothing restarted

Two `kubectl` queries confirm the recovery was in-process rather than a restart: the `pathways-proxy` init-container's `restartCount` and the JobSet's `restarts`. Both read `0`, the container and the JobSet are the same ones from before the failure.

In [ ]:
%%bash
kubectl get pod -l job-name=pw-elastic-pathways-head-0 \
  -o jsonpath='{.items[0].status.initContainerStatuses[?(@.name=="pathways-proxy")].restartCount}'
echo "  <- pathways-proxy restartCount (expect 0)"
kubectl get jobset pw-elastic -o jsonpath='{.status.restarts}'
echo "  <- jobset restarts (expect 0)"

### Trigger a second failure (optional)

The job is still running, so you can repeat the failure on a different slice to confirm recovery is repeatable. This cell runs only when `RUN_SECOND_FAILURE = True` in the parameters cell (off by default). It terminates a worker on slice 1 and waits for the second recovery.

In [ ]:
if RUN_SECOND_FAILURE:
    second_slice = "1"
    wait_for(in_safe_window,
             "safe window before second failure")
    victim = subprocess.run(
        ["kubectl", "get", "pods", "-l", f"job-name=pw-elastic-worker-{second_slice}",
         "-o", "jsonpath={.items[0].metadata.name}"],
        capture_output=True, text=True,
    ).stdout.strip()
    if not victim:
        print(f"No running worker found on slice {second_slice}; skipping.")
    else:
        print(f"Terminating {victim} (slice {second_slice})")
        subprocess.run(["kubectl", "delete", "pod", victim, "--grace-period=0", "--force"], check=True)
        wait_for(lambda t: "Restoring checkpoint from" in t,
                 "second recovery started")
else:
    print("RUN_SECOND_FAILURE is False; skipping the optional second failure.")

**What this demo does *not* show**: true elastic *degradation*, where training continues on 2 of 3 slices at reduced throughput while the third is down. That requires dynamic mesh resize in MaxText, which is not yet implemented: today the mesh is fixed at startup, so recovery always means "restore the checkpoint and wait for all slices".

## Clean up

Tear everything down to stop the meter. This deletes the JobSet and runs `terraform destroy` to remove all the infrastructure created in Step 1. The TPU slices cost ~$58/hr, so do not skip this. To remove everything completely, you can also [delete the Google Cloud project](https://cloud.google.com/resource-manager/docs/creating-managing-projects#shutting_down_projects).

### What you should see

```
Destroy complete! Resources: NN destroyed.
```

In [ ]:
%%bash
gcloud container clusters get-credentials "$CLUSTER_NAME" \
  --zone="$ZONE" --project="$PROJECT_ID" --quiet || true
kubectl delete jobset pw-elastic --ignore-not-found=true || true
terraform -chdir=terraform destroy -auto-approve

## Learn more

- **Elastic training flags**: the run is configured in `deploy/jobs/training/train.sh`, `elastic_enabled`, `elastic_timeout_seconds`, `elastic_max_retries`, `enable_single_controller` (runs training through Pathways), and `checkpoint_period=100` (frequent enough that recovery rewinds only a little).
- **Running a larger model**: during recovery the checkpoint streams through the `pathways-proxy` sidecar, capped at `memory: 100G` in `deploy/jobs/training/jobset.yaml`. Qwen3 0.6B keeps the checkpoint around 7 GiB; for a bigger model, raise that limit. See the README for details.
- **Terminal workflow**: the `Makefile` wraps every command here as `make infra`, `make deploy`, `make demo`, `make fail`, and `make destroy`.
- **Check out [MaxText documentation](https://maxtext.readthedocs.io/)**. 